# Core metrics together

This is the recommended end-to-end guide when one generation should be evaluated for Coverage, Faithfulness, and Instruction Adherence with one shared judge.

## 1. Imports

In [ ]:
from idp_eval import (
    CoverageEvaluator,
    EvaluationCase,
    EvaluationFramework,
    FaithfulnessEvaluator,
    InstructionAdherenceEvaluator,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## 2. One judge, three evaluator classes

Use application-owned configuration; values below are placeholders. `create_gateway_judge(config=...)` can supply the same evaluator API. Passing classes is concise and lets the framework inject one shared judge. Pass constructed evaluator instances instead when options such as `verbose=True` are needed.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)
framework = EvaluationFramework(
    judge=judge,
    evaluators=[
        CoverageEvaluator,
        FaithfulnessEvaluator,
        InstructionAdherenceEvaluator,
    ],
)
framework.metrics

## 3. Complete case

`input` describes the generation task for traces/reports but is not evidence for these three metrics. `context` is authoritative source evidence for Coverage and Faithfulness. `instructions` are explicit output constraints for Instruction Adherence. `output` is the generated value being evaluated. Structured dictionaries and lists are rendered generically.

In [ ]:
instructions = """
Generate exactly 3 recommendations.
Every recommendation must contain a title and rationale.
Do not recommend unapproved regions.
"""
context = {
    "approved_regions": ["US", "EU"],
    "requirements": [
        "Enable audit logging.",
        "Require administrator MFA.",
    ],
}
recommendations = [
    {"title": "US deployment", "rationale": "Use US hosting with audit logging and administrator MFA."},
    {"title": "EU deployment", "rationale": "Use EU hosting with audit logging and administrator MFA."},
    {"title": "Phased deployment", "rationale": "Start in the US, then add approved EU hosting with the same controls."},
]
case = EvaluationCase(
    case_id="generation-001",
    input="Generate deployment recommendations from the approved policy.",
    context=context,
    instructions=instructions,
    output=recommendations,
    evaluation_scope="combined",
)

## 4. Run all configured metrics

In [ ]:
results = framework.evaluate(case)
results["coverage"]
results["faithfulness"]
results["instruction_adherence"]
{name: {"score": result.score, "label": result.label} for name, result in results.items()}

## 5. Select metrics per evaluation

`framework.metrics` lists configured names. `metrics=[...]` runs only that configured subset; it does not construct new evaluators. Unknown names raise `KeyError` before judge work. Validation also applies only to selected metrics, so a case without `instructions` is valid when only Faithfulness is selected.

In [ ]:
framework.metrics
selected_results = framework.evaluate(
    case, metrics=["coverage", "faithfulness"]
)
instruction_only = framework.evaluate(
    case, metrics=["instruction_adherence"]
)
faithfulness_only_case = EvaluationCase(
    input="Summarize approved deployment facts.",
    context=context,
    output=recommendations[0],
)
faithfulness_only = framework.evaluate(
    faithfulness_only_case, metrics=["faithfulness"]
)

# This would raise KeyError before judge work; leave it unexecuted:
# framework.evaluate(case, metrics=["not_configured"])

## 6. Verbose audit details

Construct evaluator instances when item-level audit trails are useful. This changes detail verbosity, not metric semantics. See the focused [Coverage](coverage_evaluator_usage.ipynb), [Faithfulness](faithfulness_evaluator_usage.ipynb), and [Instruction Adherence](instruction_adherence_evaluator_usage.ipynb) guides for metric internals.

In [ ]:
verbose_framework = EvaluationFramework(
    judge=judge,
    evaluators=[
        CoverageEvaluator(judge, verbose=True),
        FaithfulnessEvaluator(judge, verbose=True),
        InstructionAdherenceEvaluator(judge, verbose=True),
    ],
)
verbose_results = verbose_framework.evaluate(case)
coverage_items = verbose_results["coverage"].details["items"]
faithfulness_claims = verbose_results["faithfulness"].details["claims"]
instruction_checks = verbose_results["instruction_adherence"].details["instructions"]

## 7. Combined and individual output scope

`both` evaluates the full recommendation list and each top-level item. Scope belongs to orchestration and applies uniformly to all selected metrics. Combined results are not an average of individual results; they are a separate judgment over the complete structured output.

In [ ]:
scoped_case = EvaluationCase(
    case_id="generation-001",
    input=case.input, context=context, instructions=instructions,
    output=recommendations, evaluation_scope="both",
)
scoped_results = framework.evaluate(scoped_case)
scoped_results["combined"]["coverage"]
scoped_results["combined"]["faithfulness"]
scoped_results["combined"]["instruction_adherence"]
scoped_results["individual"][0]["coverage"]
scoped_results["individual"][0]["faithfulness"]
scoped_results["individual"][0]["instruction_adherence"]

scoped_subset = framework.evaluate(
    scoped_case, metrics=["coverage", "faithfulness"]
)

Return shapes: default/combined returns `{metric: EvaluationResult}`. Individual returns `{'combined': None, 'individual': [{metric: EvaluationResult}, ...]}`. Both returns a combined mapping plus the individual mappings. With `case_id='generation-001'`, item traces use `generation-001:0`, `generation-001:1`, and so on.

## 8. Async selected metrics

Jupyter supports top-level `await`. The framework enforces one shared `max_concurrency` limit across all selected evaluator calls and scope-expanded items.

In [ ]:
async_selected = await framework.a_evaluate(
    scoped_case, metrics=["coverage", "faithfulness"], max_concurrency=4
)
async_selected["combined"]["coverage"]

## 9. Bulk evaluation

`evaluate_many()` accepts unrelated cases, preserves their order, validates only selected metrics, and lets every case keep its own scope. Its async equivalent applies one shared concurrency limit to the whole workload.

In [ ]:
second_case = EvaluationCase(
    case_id="generation-002",
    input="Summarize approved controls.",
    context=context,
    output={"summary": "Audit logging and administrator MFA are required."},
)
cases = [scoped_case, second_case]
many_results = framework.evaluate_many(
    cases, metrics=["coverage", "faithfulness"]
)
async_many_results = await framework.a_evaluate_many(
    cases, metrics=["coverage", "faithfulness"], max_concurrency=4
)

## 10. Retrieval metrics are separate

`RelevanceAtKEvaluator`, `HitRateAtKEvaluator`, `MRRAtKEvaluator`, and `NDCGAtKEvaluator` evaluate ranked `retrieved_documents` against the query in `input`. They can share a framework with these metrics, but they have a different data contract; see [Retrieval metrics usage](retrieval_metrics_usage.ipynb).

## 11. Close resources

In [ ]:
judge.close()